<a href="https://colab.research.google.com/github/21centjoe/Aleph-NELOS-1D-/blob/main/One_Bit_Codebook_1D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### One-Bit Codebook for Telemetry

A one-bit codebook in telemetry often refers to a simple mapping where a single binary value (0 or 1) represents two distinct states or messages. This is useful for highly constrained environments where bandwidth or data storage is minimal. For example:

*   `0`: "Operation Nominal"
*   `1`: "Operation Anomaly"

Or, for specific telemetry commands:

*   `0`: "Disable feature X"
*   `1`: "Enable feature X"

In [ ]:
print("--- Demonstrating Python Bitwise Operations for 'One-Bit Systems' and 'Rules' ---")

# --- 1. Basic Bitwise Operations ---
# Let's represent some 'bit codes' or flags within an integer (e.g., an 8-bit byte)
# Each bit can be considered a 'one-bit system' or a flag for a specific rule/status.

# Imagine a status byte where each bit represents a different condition:
# Bit 0 (LSB): Sensor A status (0=OK, 1=Error)
# Bit 1: Sensor B status (0=OK, 1=Error)
# Bit 2: System Mode (0=Idle, 1=Active)
# Bit 3: Warning Flag (0=None, 1=Warning)
# ...and so on.

current_status_byte = 0b00001011 # Binary for 11 (decimal)
                                # Bit 0 (1): Sensor A Error
                                # Bit 1 (1): Sensor B Error
                                # Bit 2 (0): System Mode Idle
                                # Bit 3 (1): Warning Flag Active

print(f"\nCurrent Status Byte (binary): {bin(current_status_byte)}")
print(f"Current Status Byte (decimal): {current_status_byte}")

# Define masks for individual bits (our 'one-bit systems' or 'rules')
SENSOR_A_ERROR_MASK = 0b00000001 # 2^0
SENSOR_B_ERROR_MASK = 0b00000010 # 2^1
SYSTEM_MODE_ACTIVE_MASK = 0b00000100 # 2^2
WARNING_FLAG_MASK = 0b00001000 # 2^3

# --- 2. Checking 'Rules' (Individual Bit Status) ---
# Use bitwise AND (&) to check if a specific bit is set (i.e., if a 'rule' is active)

print("\n--- Checking 'Rules' Status ---")
if (current_status_byte & SENSOR_A_ERROR_MASK):
    print("Rule: Sensor A Error is ACTIVE")
else:
    print("Rule: Sensor A is OK")

if (current_status_byte & SYSTEM_MODE_ACTIVE_MASK):
    print("Rule: System Mode is ACTIVE")
else:
    print("Rule: System Mode is IDLE")

# --- 3. Setting and Clearing 'Rules' (Bits) ---
# Use bitwise OR (|) to set a bit (activate a rule)
# Use bitwise NOT (~) and AND (&) to clear a bit (deactivate a rule)

print("\n--- Modifying 'Rules' Status ---")

# Activate 'System Mode Active' (set Bit 2)
new_status_byte = current_status_byte | SYSTEM_MODE_ACTIVE_MASK
print(f"Original: {bin(current_status_byte)} (dec: {current_status_byte})")
print(f"After activating System Mode: {bin(new_status_byte)} (dec: {new_status_byte})")

# Clear 'Warning Flag' (clear Bit 3)
# Note: ~ operates on all bits of the integer, so for specific bit clearing,
# you often combine it with a mask and AND.
clear_warning_mask = ~WARNING_FLAG_MASK # Inverts all bits
new_status_byte = new_status_byte & clear_warning_mask
print(f"After clearing Warning Flag: {bin(new_status_byte)} (dec: {new_status_byte})")

# --- 4. Toggling 'Rules' (Bits) ---
# Use bitwise XOR (^) to toggle a bit
print("\n--- Toggling 'Rules' Status ---")
toggled_status_byte = current_status_byte ^ SENSOR_A_ERROR_MASK
print(f"Original Sensor A status: {bin(current_status_byte & SENSOR_A_ERROR_MASK)}")
print(f"Toggling Sensor A error: {bin(toggled_status_byte & SENSOR_A_ERROR_MASK)}")

# --- 5. Bit Shifting for 'Linux-like' register interpretation (conceptual) ---
# This is often used to extract multi-bit fields or prepare values for registers.
# Left shift (<<) multiplies by powers of 2
# Right shift (>>) divides by powers of 2

# If a 'status register' had an 'error code' field in bits 4-7
error_code_field = 0b0101 # Example 4-bit error code
register_value = error_code_field << 4 # Shift error code into bits 4-7
print(f"\nShifted error code into register position: {bin(register_value)}")

# To extract a field (e.g., bits 4-7, assuming they are now 'error_code_field_mask')
extract_mask = 0b11110000 # Mask for bits 4-7
extracted_value = (register_value & extract_mask) >> 4
print(f"Extracted error code: {bin(extracted_value)}")

print("\nThese examples illustrate how to work with individual bits in Python, which you can use to implement 'one-bit systems' and define 'new rules' based on bit patterns. If you have specific 'bit codes' from Linux you're trying to interpret (e.g., from a file, a device driver, or network packets), please provide more details, and we can explore how to parse them using these bitwise techniques!")

--- Demonstrating Python Bitwise Operations for 'One-Bit Systems' and 'Rules' ---

Current Status Byte (binary): 0b1011
Current Status Byte (decimal): 11

--- Checking 'Rules' Status ---
Rule: Sensor A Error is ACTIVE
Rule: System Mode is IDLE

--- Modifying 'Rules' Status ---
Original: 0b1011 (dec: 11)
After activating System Mode: 0b1111 (dec: 15)
After clearing Warning Flag: 0b111 (dec: 7)

--- Toggling 'Rules' Status ---
Original Sensor A status: 0b1
Toggling Sensor A error: 0b0

Shifted error code into register position: 0b1010000
Extracted error code: 0b101

These examples illustrate how to work with individual bits in Python, which you can use to implement 'one-bit systems' and define 'new rules' based on bit patterns. If you have specific 'bit codes' from Linux you're trying to interpret (e.g., from a file, a device driver, or network packets), please provide more details, and we can explore how to parse them using these bitwise techniques!


### Applying 'Overlays' to Linux-like Bit Codes in Python

When working with low-level telemetry or status registers from a Linux system, a single multi-bit integer often contains various pieces of information, each occupying specific bit ranges. We can use Python to 'overlay' different interpretations onto this single integer by defining masks and shift operations, effectively decoding these 'bit codes' into meaningful telemetry signals or status flags.

In [ ]:
print("\n--- Interpreting a Composite 'Linux-like' Status Word ---")

# Let's imagine a 32-bit status word received from a Linux system or device driver.
# This single integer contains several pieces of telemetry information.
# Example: 0x8000_1234 (hex representation)
# Bit 31: Global Error Flag (1=Error, 0=OK)
# Bits 24-27: System Mode (0=Idle, 1=Active, 2=Config, ...)
# Bits 16-23: Temperature Sensor Value (8-bit integer)
# Bits 0-7: Device ID (8-bit integer)

composite_status_word = 0b10000000000000001001001101010100 # Example 32-bit value
# In hex: 0x80009354
# Global Error: Bit 31 is 1
# System Mode: Bits 24-27 (0b0000) = 0 (Idle)
# Temperature: Bits 16-23 (0b10010011) = 147 (decimal)
# Device ID: Bits 0-7 (0b01010100) = 84 (decimal)

print(f"Composite Status Word (binary): {bin(composite_status_word)}")
print(f"Composite Status Word (hex): {hex(composite_status_word)}")

# --- Define 'Overlays' (Masks and Shifts) for each field ---

# Overlay 1: Global Error Flag (1-bit) - most significant bit
GLOBAL_ERROR_FLAG_MASK = 0x80000000 # Bit 31 (2^31)

# Overlay 2: System Mode (4-bit field)
SYSTEM_MODE_MASK = 0x0F000000 # Bits 24-27
SYSTEM_MODE_SHIFT = 24

# Overlay 3: Temperature Sensor Value (8-bit field)
TEMPERATURE_MASK = 0x00FF0000 # Bits 16-23
TEMPERATURE_SHIFT = 16

# Overlay 4: Device ID (8-bit field)
DEVICE_ID_MASK = 0x000000FF # Bits 0-7
DEVICE_ID_SHIFT = 0

# --- Apply the 'Overlays' to decode the status word ---
print("\n--- Decoded Telemetry Signals (Applying Overlays) ---")

is_global_error = (composite_status_word & GLOBAL_ERROR_FLAG_MASK) != 0
print(f"Global Error Detected: {is_global_error}")

system_mode = (composite_status_word & SYSTEM_MODE_MASK) >> SYSTEM_MODE_SHIFT
# Map system mode integer to a human-readable string
system_mode_map = {0: 'Idle', 1: 'Active', 2: 'Configuration', 3: 'Sleep'}
print(f"System Mode: {system_mode_map.get(system_mode, f'Unknown ({system_mode})')}")

temperature_value = (composite_status_word & TEMPERATURE_MASK) >> TEMPERATURE_SHIFT
print(f"Temperature Value: {temperature_value} C")

device_id = (composite_status_word & DEVICE_ID_MASK) >> DEVICE_ID_SHIFT
print(f"Device ID: {device_id}")

# --- Handling an 'Initialized Telemetry Signal' (1-bit) ---
# Let's say we have another 8-bit register where the 1st bit (Bit 0) signifies 'initialized'.
initialized_register = 0b00000001 # Value indicating initialized
INITIALIZED_FLAG = 0b00000001 # Mask for the initialized bit

is_telemetry_initialized = (initialized_register & INITIALIZED_FLAG) != 0
print(f"\nIs Telemetry System Initialized (1-bit signal): {is_telemetry_initialized}")

# --- Handling a 'Terminal Telemetry Signal' (1-bit) ---
# Let's say the 2nd bit (Bit 1) in the same register signifies 'terminal state'.
terminal_register = 0b00000010 # Value indicating terminal state
TERMINAL_FLAG = 0b00000010 # Mask for the terminal bit

is_telemetry_terminal = (terminal_register & TERMINAL_FLAG) != 0
print(f"Is Telemetry System in Terminal State (1-bit signal): {is_telemetry_terminal}")


--- Interpreting a Composite 'Linux-like' Status Word ---
Composite Status Word (binary): 0b10000000000000001001001101010100
Composite Status Word (hex): 0x80009354

--- Decoded Telemetry Signals (Applying Overlays) ---
Global Error Detected: True
System Mode: Idle
Temperature Value: 0 C
Device ID: 84

Is Telemetry System Initialized (1-bit signal): True
Is Telemetry System in Terminal State (1-bit signal): True


This example demonstrates how Python can be used to apply specific 'overlays' (mask and shift operations) to a single integer representing a collection of bit codes, breaking it down into individual, meaningful telemetry signals of various bit-widths, including single-bit flags for states like 'initialized' or 'terminal'. This approach is highly effective for interpreting data formats common in low-level system interactions.

### Simulating Multi-bit and Single-bit Telemetry Signals in Python

Since direct interaction with the Linux kernel is not possible in this environment, we can simulate the handling of various bit-width telemetry signals using Python integers and bitwise operations. This allows us to represent, extract, and interpret data fields of different sizes, including individual bits for critical flags like 'initialized' or 'terminal' signals.

In [ ]:
print("\n--- Simulating Multi-bit Telemetry Signals (64-bit, 32-bit, 8-bit) ---")

# Imagine a 64-bit telemetry packet or status word
# For demonstration, let's create a hypothetical 64-bit integer.
# In a real scenario, this might come from reading a device register or network data.
telemetry_64bit_signal = 0xDEADBEEF01234567 # A large hexadecimal number representing 64 bits
print(f"Original 64-bit Signal: {hex(telemetry_64bit_signal)}")

# --- Extracting a 32-bit sub-signal ---
# Let's say a 32-bit error code is in the lower 32 bits.
# We can mask out the upper bits or just take the lower portion if it's already aligned.
error_code_32bit_mask = 0xFFFFFFFF # Mask for the lower 32 bits
error_code_32bit = telemetry_64bit_signal & error_code_32bit_mask
print(f"Extracted 32-bit Error Code: {hex(error_code_32bit)}")

# --- Extracting an 8-bit sub-signal ---
# Let's say a device ID is in bits 8-15 (the second byte from the right).
device_id_8bit_mask = 0xFF00 # Mask for bits 8-15
device_id_8bit = (telemetry_64bit_signal & device_id_8bit_mask) >> 8 # Shift right by 8 to get the value
print(f"Extracted 8-bit Device ID: {hex(device_id_8bit)}")

# --- Extracting a 1-bit flag (e.g., initialized status) ---
# Let's say bit 0 (LSB) indicates 'initialized' status (1=initialized, 0=not initialized)
initialized_flag_mask = 0x01 # Mask for bit 0
is_initialized = (telemetry_64bit_signal & initialized_flag_mask) != 0
print(f"Is Initialized Flag (Bit 0): {is_initialized}")

# --- Packing Data (creating a multi-bit signal from smaller parts) ---
# You can also combine smaller signals into a larger one.
status_8bit = 0xAA # Some 8-bit status
command_8bit = 0x55 # Some 8-bit command

# Combine them into a 16-bit word
combined_16bit = (status_8bit << 8) | command_8bit
print(f"Combined 16-bit word (from 2x8-bit): {hex(combined_16bit)}")


--- Simulating Multi-bit Telemetry Signals (64-bit, 32-bit, 8-bit) ---
Original 64-bit Signal: 0xdeadbeef01234567
Extracted 32-bit Error Code: 0x1234567
Extracted 8-bit Device ID: 0x45
Is Initialized Flag (Bit 0): True
Combined 16-bit word (from 2x8-bit): 0xaa55


### Handling One-Bit Telemetry Signals for Specific Applications

For critical signals like 'initialized' or 'terminal' status, a single bit is highly efficient. We can define specific bit positions or simple integer values to represent these states.

In [ ]:
print("\n--- One-Bit Telemetry Signals for Initialization/Termination ---")

# Define simple one-bit codes directly
INITIALIZED_SIGNAL = 1
NOT_INITIALIZED_SIGNAL = 0

TERMINAL_SIGNAL = 1
NOT_TERMINAL_SIGNAL = 0

# Scenario: Receiving a 1-bit 'initialized' status
received_init_status = INITIALIZED_SIGNAL # or NOT_INITIALIZED_SIGNAL
if received_init_status == INITIALIZED_SIGNAL:
    print(f"Received Initialization Status: System is INITIALIZED ({received_init_status})")
else:
    print(f"Received Initialization Status: System is NOT INITIALIZED ({received_init_status})")

# Scenario: Receiving a 1-bit 'terminal' status
received_term_status = NOT_TERMINAL_SIGNAL # or TERMINAL_SIGNAL
if received_term_status == TERMINAL_SIGNAL:
    print(f"Received Terminal Status: System is at TERMINAL STATE ({received_term_status})")
else:
    print(f"Received Terminal Status: System is NOT at TERMINAL STATE ({received_term_status})")

# You can also use a bit within a larger status byte for these single-bit flags
# Example: Status byte where Bit 0 is 'Initialized' and Bit 1 is 'Terminal'
STATUS_BYTE_INIT_MASK = 0x01
STATUS_BYTE_TERM_MASK = 0x02

current_device_status = 0b00000001 # Only initialized

if (current_device_status & STATUS_BYTE_INIT_MASK) != 0:
    print("From status byte: Device is INITIALIZED")
else:
    print("From status byte: Device is NOT INITIALIZED")

if (current_device_status & STATUS_BYTE_TERM_MASK) != 0:
    print("From status byte: Device is in TERMINAL STATE")
else:
    print("From status byte: Device is NOT in TERMINAL STATE")


--- One-Bit Telemetry Signals for Initialization/Termination ---
Received Initialization Status: System is INITIALIZED (1)
Received Terminal Status: System is NOT at TERMINAL STATE (0)
From status byte: Device is INITIALIZED
From status byte: Device is NOT in TERMINAL STATE


These examples provide a foundation for working with various bit-width signals in Python. If you have specific data formats, message structures, or more concrete 'bitwise codes' you're trying to interpret from a Linux-based system (e.g., logs, custom network protocols, or specific device driver outputs), please share them! I can then help you write more targeted Python code to parse and understand them.

In [ ]:
# Define a simple one-bit codebook
# This dictionary maps the binary value (0 or 1) to a descriptive message.
telemetry_codebook = {
    0: "Status: Nominal",
    1: "Status: Anomaly Detected"
}

# Example usage:
telemetry_status = 0 # Let's say we receive a '0' from a sensor
print(f"Received telemetry code '{telemetry_status}': {telemetry_codebook[telemetry_status]}")

telemetry_status = 1 # Now, let's simulate an anomaly
print(f"Received telemetry code '{telemetry_status}': {telemetry_codebook[telemetry_status]}")

Received telemetry code '0': Status: Nominal
Received telemetry code '1': Status: Anomaly Detected


In [ ]:
# Define the inverted one-bit codebook
# Here, 0 represents 'Anomaly Detected' and 1 represents 'Nominal'.
telemetry_codebook_inverted = {
    0: "Status: Anomaly Detected",
    1: "Status: Nominal"
}

print("\n--- Using Inverted Codebook ---")
# Example usage with the inverted codebook:
inverted_telemetry_status = 0 # Let's say we receive a '0'
print(f"Received telemetry code '{inverted_telemetry_status}': {telemetry_codebook_inverted[inverted_telemetry_status]}")

inverted_telemetry_status = 1 # Now, let's simulate a nominal state
print(f"Received telemetry code '{inverted_telemetry_status}': {telemetry_codebook_inverted[inverted_telemetry_status]}")


--- Using Inverted Codebook ---
Received telemetry code '0': Status: Anomaly Detected
Received telemetry code '1': Status: Nominal


In [ ]:
# Define a simple one-bit codebook
# This dictionary maps the binary value (0 or 1) to a descriptive message.
telemetry_codebook = {
    0: "Status: Nominal",
    1: "Status: Anomaly Detected"
}

# Define the inverted one-bit codebook
# Here, 0 represents 'Anomaly Detected' and 1 represents 'Nominal'.
telemetry_codebook_inverted = {
    0: "Status: Anomaly Detected",
    1: "Status: Nominal"
}

def decode_telemetry_list(codes_list, codebook):
    """
    Decodes a list of telemetry status codes using a given codebook.

    Args:
        codes_list (list): A list of integer telemetry codes (0 or 1).
        codebook (dict): A dictionary mapping telemetry codes to their meanings.

    Returns:
        list: A list of decoded messages corresponding to the input codes.
    """
    decoded_messages = []
    for code in codes_list:
        decoded_messages.append(codebook.get(code, f"Unknown Code: {code}"))
    return decoded_messages

# Example usage with the original codebook
telemetry_codes_to_decode = [0, 1, 0, 0, 1]
decoded_results = decode_telemetry_list(telemetry_codes_to_decode, telemetry_codebook)

print("\n--- Decoding with Original Codebook ---")
for i, code in enumerate(telemetry_codes_to_decode):
    print(f"Code: {code} -> Message: {decoded_results[i]}")

# Example usage with the inverted codebook
inverted_telemetry_codes_to_decode = [1, 0, 1, 0, 0]
inverted_decoded_results = decode_telemetry_list(inverted_telemetry_codes_to_decode, telemetry_codebook_inverted)

print("\n--- Decoding with Inverted Codebook ---")
for i, code in enumerate(inverted_telemetry_codes_to_decode):
    print(f"Code: {code} -> Message: {inverted_decoded_results[i]}")

NameError: name 'telemetry_codebook' is not defined

In [ ]:
class TelemetryStatusManager:
    """
    Manages a telemetry status byte using bitwise operations to apply and check rules.
    Each rule corresponds to a specific bit in the status byte.
    """
    def __init__(self, initial_status_byte=0):
        self._status_byte = initial_status_byte
        # _rules stores a mapping of rule_name to its (mask, description)
        self._rules = {}
        print(f"TelemetryStatusManager initialized with status: {bin(self._status_byte)}")

    def add_rule(self, rule_name, mask, description):
        """
        Adds a new rule to the manager.

        Args:
            rule_name (str): A unique name for the rule (e.g., 'SENSOR_A_ERROR').
            mask (int): The bitmask (power of 2) corresponding to this rule.
            description (str): A human-readable description of the rule.
        """
        if not isinstance(mask, int) or (mask & (mask - 1) != 0 and mask != 0):
            raise ValueError("Mask must be a power of 2 (representing a single bit).")
        if rule_name in self._rules:
            print(f"Warning: Rule '{rule_name}' already exists and will be overwritten.")
        self._rules[rule_name] = {'mask': mask, 'description': description}
        print(f"Rule '{rule_name}' added (Mask: {bin(mask)}).")

    def get_status_byte(self):
        """
        Returns the current telemetry status byte.
        """
        return self._status_byte

    def check_rule(self, rule_name):
        """
        Checks if a specific rule (bit) is currently active in the status byte.

        Args:
            rule_name (str): The name of the rule to check.

        Returns:
            bool: True if the rule is active, False otherwise.
        """
        if rule_name not in self._rules:
            print(f"Error: Rule '{rule_name}' not found.")
            return False
        mask = self._rules[rule_name]['mask']
        return (self._status_byte & mask) != 0

    def set_rule(self, rule_name):
        """
        Activates a specific rule (sets the corresponding bit) in the status byte.

        Args:
            rule_name (str): The name of the rule to activate.
        """
        if rule_name not in self._rules:
            print(f"Error: Rule '{rule_name}' not found. Cannot set.")
            return
        mask = self._rules[rule_name]['mask']
        self._status_byte |= mask
        print(f"Rule '{rule_name}' set. New status: {bin(self._status_byte)}")

    def clear_rule(self, rule_name):
        """
        Deactivates a specific rule (clears the corresponding bit) in the status byte.

        Args:
            rule_name (str): The name of the rule to deactivate.
        """
        if rule_name not in self._rules:
            print(f"Error: Rule '{rule_name}' not found. Cannot clear.")
            return
        mask = self._rules[rule_name]['mask']
        self._status_byte &= ~mask
        print(f"Rule '{rule_name}' cleared. New status: {bin(self._status_byte)}")

    def toggle_rule(self, rule_name):
        """
        Toggles the state of a specific rule (bit) in the status byte.

        Args:
            rule_name (str): The name of the rule to toggle.
        """
        if rule_name not in self._rules:
            print(f"Error: Rule '{rule_name}' not found. Cannot toggle.")
            return
        mask = self._rules[rule_name]['mask']
        self._status_byte ^= mask
        print(f"Rule '{rule_name}' toggled. New status: {bin(self._status_byte)}")

    def get_active_rules(self):
        """
        Returns a list of descriptions for all currently active rules.
        """
        active_rule_descriptions = []
        for rule_name, rule_data in self._rules.items():
            if (self._status_byte & rule_data['mask']) != 0:
                active_rule_descriptions.append(rule_data['description'])
        return active_rule_descriptions

# --- Demonstration of the TelemetryStatusManager Class ---
print("\n--- TelemetryStatusManager Demonstration ---")

# 1. Initialize the manager
tsm = TelemetryStatusManager(initial_status_byte=0b00001011) # Initial status with Sensor A Error, Sensor B Error, Warning Flag

# 2. Add some rules
tsm.add_rule('SENSOR_A_ERROR', 0b00000001, 'Sensor A reports an error.')
tsm.add_rule('SENSOR_B_ERROR', 0b00000010, 'Sensor B reports an error.')
tsm.add_rule('SYSTEM_MODE_ACTIVE', 0b00000100, 'System operating in active mode.')
tsm.add_rule('WARNING_FLAG', 0b00001000, 'A general system warning has been triggered.')
tsm.add_rule('DATA_OVERLOAD', 0b00010000, 'Data buffer is overloaded.')

# 3. Check initial status for specific rules
print("\n--- Initial Rule Checks ---")
print(f"Sensor A Error active: {tsm.check_rule('SENSOR_A_ERROR')}")
print(f"System Mode Active: {tsm.check_rule('SYSTEM_MODE_ACTIVE')}")
print(f"Warning Flag active: {tsm.check_rule('WARNING_FLAG')}")

# 4. Get and print all active rules
print("\n--- Currently Active Rules ---")
active_rules = tsm.get_active_rules()
if active_rules:
    for rule_desc in active_rules:
        print(f"- {rule_desc}")
else:
    print("No rules currently active.")

# 5. Modify status (set, clear, toggle rules)
print("\n--- Modifying Rules ---")
tsm.set_rule('SYSTEM_MODE_ACTIVE') # Activate system mode
tsm.clear_rule('WARNING_FLAG')      # Clear the warning
tsm.toggle_rule('SENSOR_A_ERROR')   # Toggle Sensor A error (should go from active to inactive)
tsm.set_rule('DATA_OVERLOAD')      # Trigger data overload

# 6. Check status after modifications
print("\n--- Status After Modifications ---")
print(f"Current status byte: {bin(tsm.get_status_byte())}")
print(f"Sensor A Error active: {tsm.check_rule('SENSOR_A_ERROR')}")
print(f"System Mode Active: {tsm.check_rule('SYSTEM_MODE_ACTIVE')}")
print(f"Warning Flag active: {tsm.check_rule('WARNING_FLAG')}")
print(f"Data Overload active: {tsm.check_rule('DATA_OVERLOAD')}")

# 7. Get and print all active rules again
print("\n--- Active Rules After Modifications ---")
active_rules = tsm.get_active_rules()
if active_rules:
    for rule_desc in active_rules:
        print(f"- {rule_desc}")
else:
    print("No rules currently active.")


--- TelemetryStatusManager Demonstration ---
TelemetryStatusManager initialized with status: 0b1011
Rule 'SENSOR_A_ERROR' added (Mask: 0b1).
Rule 'SENSOR_B_ERROR' added (Mask: 0b10).
Rule 'SYSTEM_MODE_ACTIVE' added (Mask: 0b100).
Rule 'WARNING_FLAG' added (Mask: 0b1000).
Rule 'DATA_OVERLOAD' added (Mask: 0b10000).

--- Initial Rule Checks ---
Sensor A Error active: True
System Mode Active: False
Warning Flag active: True

--- Currently Active Rules ---
- Sensor A reports an error.
- Sensor B reports an error.
- A general system warning has been triggered.

--- Modifying Rules ---
Rule 'SYSTEM_MODE_ACTIVE' set. New status: 0b1111
Rule 'WARNING_FLAG' cleared. New status: 0b111
Rule 'SENSOR_A_ERROR' toggled. New status: 0b110
Rule 'DATA_OVERLOAD' set. New status: 0b10110

--- Status After Modifications ---
Current status byte: 0b10110
Sensor A Error active: False
System Mode Active: True
Warning Flag active: False
Data Overload active: True

--- Active Rules After Modifications ---
- Sen